# Churn Feature Engineering 07-2 - Leakage-safe Pipeline

이 노트북은 train 기준으로 quantile threshold를 fit한 뒤 test에 transform하는 leakage-safe feature engineering 실험이다. First Try를 실행하지 않고도 바로 pipeline 기반 성능 비교를 수행할 수 있다.


## 피쳐 설명

이 노트북은 `07-1`의 함수형 feature engineering을 Pipeline 안에서 사용할 수 있는 leakage-safe transformer로 재구성한다. quantile 기준값은 `fit()`에서 train 데이터로만 계산하고, valid/test에는 저장된 기준값을 그대로 적용한다.

| 구분 | 피쳐명 | 설명 |
| --- | --- | --- |
| 사용 강도 | `watch_time_per_session` | 세션당 평균 시청 시간 |
| 사용 강도 | `weekly_watch_session_intensity` | 주당 시청 시간과 주당 시청 세션 수를 곱한 소비 강도 |
| 사용 강도 | `session_count_per_week_session` | 주당 시청 세션 대비 접속 세션 수 |
| 계정 연령 대비 사용 | `watch_time_per_account_age` | 계정 연령 대비 주당 평균 시청 시간 |
| 계정 연령 대비 사용 | `session_per_account_age` | 계정 연령 대비 접속 세션 수 |
| 계정 연령 대비 사용 | `weekly_session_per_account_age` | 계정 연령 대비 주당 시청 세션 수 |
| 완료 행동 | `completion_per_watch_time` | 시청 시간 대비 콘텐츠 완료율 |
| 완료 행동 | `completed_watch_minutes_proxy` | 완료율을 반영한 완료 시청 시간 proxy |
| 완료 행동 | `incomplete_watch_minutes_proxy` | 미완료율을 반영한 미완료 시청 시간 proxy |
| 완료 flag | `low_completion_flag` | 완료율이 train 1사분위수 이하인 사용자 flag |
| 완료 flag | `high_completion_flag` | 완료율이 train 3사분위수 이상인 사용자 flag |
| 추천 반응 | `recommendation_completion_gap` | 추천 클릭률과 완료율의 차이 |
| 추천 반응 | `recommendation_completion_ratio` | 완료율 대비 추천 클릭률 비율 |
| 추천 flag | `low_recommendation_response_flag` | 추천 클릭률이 train 1사분위수 이하인 사용자 flag |
| 추천 flag | `high_recommendation_response_flag` | 추천 클릭률이 train 3사분위수 이상인 사용자 flag |
| 추천/완료 결합 | `low_interest_flag` | 추천 반응과 완료율이 모두 낮은 사용자 flag |
| 비활성 | `inactive_ratio_to_account_age` | 계정 연령 대비 미접속 기간 비율 |
| 비활성 flag | `high_inactivity_flag` | 미접속 기간이 train 3사분위수 이상인 사용자 flag |
| 사용량 flag | `low_watch_flag` | 시청 시간이 train 1사분위수 이하인 사용자 flag |
| 사용량 flag | `low_session_flag` | 시청 세션 수가 train 1사분위수 이하인 사용자 flag |
| 결합 위험 | `inactive_low_watch_flag` | 높은 비활성과 낮은 시청량이 동시에 나타나는 사용자 flag |
| Engagement | `engagement_score` | 시청량, 세션, 완료율, 추천 반응, 최근 접속성을 결합한 0~100 점수 |
| Engagement | `low_engagement_flag` | engagement score가 낮은 사용자 flag |
| 범주 조합 | `basic_mobile_flag` | Basic 요금제와 Mobile 중심 사용 조합 |
| 범주 조합 | `basic_inactive_flag` | Basic 요금제와 높은 비활성 조합 |
| 범주 조합 | `mobile_low_completion_flag` | Mobile 중심 사용과 낮은 완료율 조합 |
| 범주 조합 | `plan_device_combo` | 요금제와 주 사용 기기 조합 |
| 범주 조합 | `plan_payment_combo` | 요금제와 결제 수단 조합 |
| 범주 조합 | `genre_time_combo` | 선호 장르와 주 이용 시간대 조합 |
| 범주 조합 | `region_plan_combo` | 지역과 요금제 조합 |
| 범주 조합 | `source_genre_combo` | 추천/유입 경로와 선호 장르 조합 |
| 위험 집계 | `high_risk_behavior_count` | 주요 위험 flag가 몇 개 켜졌는지 합산한 값 |
| 위험 집계 | `multi_risk_flag` | 위험 flag가 3개 이상인 사용자 flag |
| 위험 집계 | `critical_risk_flag` | 위험 flag가 5개 이상인 사용자 flag |
| 캠페인 세그먼트 | `retention_action_segment` | 위험 신호 조합에 따라 retention action 후보를 나눈 범주 |

주의: 이 노트북의 핵심은 피쳐 자체보다 train/test split 이후 Pipeline 안에서 기준값을 학습해 데이터 누수를 줄이는 구조다.

## 1. 라이브러리 로드

공통 import와 평가 helper를 로드한다. 이 노트북의 핵심 비교는 Logistic Regression, RandomForest, GradientBoosting, optional LightGBM에서 원본 피처와 behavior feature engineering 피처를 같은 조건으로 비교하는 것이다.


In [1]:
from pathlib import Path
import warnings

import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import randint, uniform, loguniform

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings('ignore')

RANDOM_STATE = 42
TEST_SIZE = 0.2
N_SPLITS = 5

font_candidates = ['AppleGothic', 'NanumGothic', 'Malgun Gothic']
available_fonts = {font.name for font in fm.fontManager.ttflist}
font_name = next((font for font in font_candidates if font in available_fonts), None)
if font_name:
    plt.rcParams['font.family'] = font_name
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style='whitegrid')


def make_ohe(**kwargs):
    try:
        return OneHotEncoder(handle_unknown='ignore', sparse_output=False, **kwargs)
    except TypeError:
        return OneHotEncoder(handle_unknown='ignore', sparse=False, **kwargs)

## 2. 데이터 로드와 Split

이전 노트북과 동일하게 v2 데이터를 사용한다. `user_id`는 식별자라 모델 피처에서는 제외하고, `churned`를 타겟으로 둔다.


In [2]:
data_candidates = [
    Path('data/user_behavior_50000/netflix_user_behavior_churn_50000v2.csv'),
    Path('../data/user_behavior_50000/netflix_user_behavior_churn_50000v2.csv'),
]
data_path = next(path for path in data_candidates if path.exists())

df = pd.read_csv(data_path)
X_original = df.drop(columns=['user_id', 'churned'])
y = df['churned']

X_train_original, X_test_original, y_train, y_test = train_test_split(
    X_original,
    y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE,
)

print('Data path:', data_path)
print('Train shape:', X_train_original.shape, 'Test shape:', X_test_original.shape)
print('Train churn ratio:', round(y_train.mean(), 4))
print('Test churn ratio:', round(y_test.mean(), 4))
X_train_original.head()

Data path: ../data/user_behavior_50000/netflix_user_behavior_churn_50000v2.csv
Train shape: (40000, 18) Test shape: (10000, 18)
Train churn ratio: 0.2093
Test churn ratio: 0.2093


,age,gender,region,subscription_type,payment_method,primary_device,account_age_months,favorite_genre,time_of_day,recommendation_source,session_count,avg_watch_time_minutes_per_week,watch_sessions_per_week,completion_rate,avg_rating_given,app_rating,recommendation_click_rate,days_since_last_login
46657,30,Female,Europe,Standard,Debit Card,Mobile,7,Drama,Night,Trending,1,118,2,65,3,4,19,1
20332,38,Male,South America,Basic,Debit Card,Tablet,1,Drama,Evening,Trending,1,158,5,65,4,4,46,21
18319,38,Female,Europe,Premium,Credit Card,Smart TV,33,Drama,Evening,Algorithm,1,249,4,89,5,5,69,11
1776,39,Male,Europe,Premium,Credit Card,Smart TV,86,Sci-Fi,Evening,Algorithm,1,279,6,90,5,4,63,1
34177,71,Male,Europe,Basic,Paypal,Smart TV,115,Documentary,Night,Friend,1,254,3,79,4,4,35,5


## 3. 평가 함수

공통 평가 helper를 정의한다. 뒤쪽 campaign 평가 함수에서 PR AUC, top-k churn capture, lift까지 확장해서 비교한다.


In [3]:
def get_scores(model, X_eval):
    if hasattr(model, 'predict_proba'):
        return model.predict_proba(X_eval)[:, 1]
    raw_scores = model.decision_function(X_eval)
    return (raw_scores - raw_scores.min()) / (raw_scores.max() - raw_scores.min())


def evaluate_predictions(y_true, y_score, threshold=0.5):
    y_pred = (y_score >= threshold).astype(int)
    return {
        'threshold': threshold,
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_true, y_score),
        'pr_auc': average_precision_score(y_true, y_score),
        'confusion_matrix': confusion_matrix(y_true, y_pred),
    }


def threshold_table(y_true, y_score, thresholds=np.arange(0.05, 0.96, 0.01)):
    rows = []
    for threshold in thresholds:
        row = evaluate_predictions(y_true, y_score, threshold=threshold)
        rows.append({k: v for k, v in row.items() if k != 'confusion_matrix'})
    return pd.DataFrame(rows)


def summarize_model_result(name, model, X_eval, y_eval):
    y_score = get_scores(model, X_eval)
    base_metrics = evaluate_predictions(y_eval, y_score, threshold=0.5)
    threshold_result = threshold_table(y_eval, y_score)
    best_f1_row = threshold_result.loc[threshold_result['f1'].idxmax()]
    recall_target = 0.70
    recall_candidates = threshold_result[threshold_result['recall'] >= recall_target]
    best_recall_row = recall_candidates.loc[recall_candidates['f1'].idxmax()] if len(recall_candidates) else best_f1_row

    base_row = {k: v for k, v in base_metrics.items() if k != 'confusion_matrix'}
    base_row['feature_set'] = name
    base_row['selection'] = 'threshold_0.50'

    tuned_row = best_recall_row.to_dict()
    tuned_row['feature_set'] = name
    tuned_row['selection'] = 'best_f1_with_recall>=0.70'

    return {
        'score': y_score,
        'base_row': base_row,
        'tuned_row': tuned_row,
        'threshold_result': threshold_result,
        'base_confusion_matrix': base_metrics['confusion_matrix'],
        'selected_confusion_matrix': evaluate_predictions(
            y_eval, y_score, threshold=float(best_recall_row['threshold'])
        )['confusion_matrix'],
    }

## 4. 전처리와 Stacking 모델 생성 함수

feature set마다 수치형/범주형 컬럼이 달라질 수 있으므로, train 데이터에서 생성된 컬럼 schema 기준으로 전처리기를 자동 구성한다.


In [4]:
def build_preprocessor(X_train):
    numeric_features = X_train.select_dtypes(include='number').columns.tolist()
    categorical_features = X_train.select_dtypes(include=['object', 'string', 'category']).columns.tolist()
    return ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), numeric_features),
            ('cat', make_ohe(), categorical_features),
        ],
        remainder='drop',
        verbose_feature_names_out=False,
    )


def make_base_estimators(y_train):
    estimators = []

    estimators.append((
        'lr',
        LogisticRegression(
            C=0.1,
            class_weight='balanced',
            max_iter=2000,
            random_state=RANDOM_STATE,
        ),
    ))

    try:
        from lightgbm import LGBMClassifier
        estimators.append((
            'lightgbm',
            LGBMClassifier(
                n_estimators=500,
                learning_rate=0.03,
                num_leaves=31,
                subsample=0.9,
                colsample_bytree=0.9,
                class_weight='balanced',
                random_state=RANDOM_STATE,
                n_jobs=-1,
                verbose=-1,
            ),
        ))
        print('[candidate] LightGBM added')
    except Exception as exc:
        print('[candidate] LightGBM unavailable:', exc)

    try:
        from xgboost import XGBClassifier
        neg, pos = np.bincount(y_train)
        estimators.append((
            'xgboost',
            XGBClassifier(
                n_estimators=500,
                learning_rate=0.03,
                max_depth=4,
                subsample=0.9,
                colsample_bytree=0.9,
                eval_metric='logloss',
                scale_pos_weight=neg / pos,
                random_state=RANDOM_STATE,
                n_jobs=-1,
            ),
        ))
        print('[candidate] XGBoost added')
    except Exception as exc:
        print('[candidate] XGBoost unavailable:', exc)

    return estimators


def make_stacking_pipeline(X_train, y_train):
    preprocessor = build_preprocessor(X_train)
    stacking = StackingClassifier(
        estimators=make_base_estimators(y_train),
        final_estimator=LogisticRegression(
            max_iter=1000,
            class_weight='balanced',
            random_state=RANDOM_STATE,
        ),
        stack_method='predict_proba',
        cv=StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE),
        n_jobs=-1,
    )
    return Pipeline([
        ('preprocess', preprocessor),
        ('model', stacking),
    ])

## 11. Leakage-safe Feature Engineering Transformer

앞의 First Try는 간단한 함수형 실험이었다. 이번 섹션은 실제 모델링 파이프라인에 넣을 수 있도록 `BaseEstimator`, `TransformerMixin` 기반 transformer로 다시 구성한다.

핵심 원칙은 다음과 같다.

- 원본 컬럼은 삭제하거나 덮어쓰지 않고, 새 파생변수만 추가한다.
- quantile 기반 기준값은 `fit()` 단계에서 train 데이터로만 계산한다.
- test 데이터에는 train에서 저장한 기준값을 그대로 적용한다.
- 피처는 churn 여부를 직접 맞히기보다 risk score ranking과 retention campaign action으로 연결될 수 있는 행동 신호 위주로 만든다.


In [5]:
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier


class ChurnBehaviorFeatureEngineer(BaseEstimator, TransformerMixin):
    """Netflix-style churn risk feature engineering transformer.

    이 transformer는 원본 컬럼을 보존하고 파생변수만 추가한다.
    quantile 기반 flag 기준은 fit 단계에서 train 데이터로만 저장하므로
    train/test split 이후 Pipeline 안에서 사용해도 데이터 누수를 줄일 수 있다.
    """

    def __init__(self, eps=1e-6):
        self.eps = eps

    def fit(self, X, y=None):
        X_fit = X.copy()

        # Train 데이터에서만 기준값을 학습한다. Test에는 이 값을 그대로 적용한다.
        self.watch_time_low_q_ = X_fit['avg_watch_time_minutes_per_week'].quantile(0.25)
        self.watch_time_high_q_ = X_fit['avg_watch_time_minutes_per_week'].quantile(0.75)
        self.sessions_low_q_ = X_fit['watch_sessions_per_week'].quantile(0.25)
        self.sessions_high_q_ = X_fit['watch_sessions_per_week'].quantile(0.75)
        self.completion_low_q_ = X_fit['completion_rate'].quantile(0.25)
        self.completion_high_q_ = X_fit['completion_rate'].quantile(0.75)
        self.reco_click_low_q_ = X_fit['recommendation_click_rate'].quantile(0.25)
        self.reco_click_high_q_ = X_fit['recommendation_click_rate'].quantile(0.75)
        self.inactive_high_q_ = X_fit['days_since_last_login'].quantile(0.75)
        self.account_age_low_q_ = X_fit['account_age_months'].quantile(0.25)

        # Engagement score 정규화를 위한 train 기준 범위.
        self.watch_time_p95_ = max(X_fit['avg_watch_time_minutes_per_week'].quantile(0.95), self.eps)
        self.session_p95_ = max(X_fit['watch_sessions_per_week'].quantile(0.95), self.eps)
        self.inactive_p95_ = max(X_fit['days_since_last_login'].quantile(0.95), self.eps)

        return self

    def transform(self, X):
        X_fe = X.copy()
        eps = self.eps

        # 1) Usage intensity features: 고객이 한 세션에서 얼마나 깊게 소비하는지 본다.
        X_fe['watch_time_per_session'] = (
            X_fe['avg_watch_time_minutes_per_week']
            / (X_fe['watch_sessions_per_week'] + eps)
        )
        X_fe['weekly_watch_session_intensity'] = (
            X_fe['avg_watch_time_minutes_per_week']
            * X_fe['watch_sessions_per_week']
        )
        X_fe['session_count_per_week_session'] = (
            X_fe['session_count']
            / (X_fe['watch_sessions_per_week'] + eps)
        )

        # 2) Account age 대비 usage: 가입 기간 대비 현재 사용 강도가 낮은지 확인한다.
        X_fe['watch_time_per_account_age'] = (
            X_fe['avg_watch_time_minutes_per_week']
            / (X_fe['account_age_months'] + 1)
        )
        X_fe['session_per_account_age'] = (
            X_fe['session_count']
            / (X_fe['account_age_months'] + 1)
        )
        X_fe['weekly_session_per_account_age'] = (
            X_fe['watch_sessions_per_week']
            / (X_fe['account_age_months'] + 1)
        )

        # 3) Completion behavior: 많이 보더라도 끝까지 보지 않으면 만족도가 낮을 수 있다.
        X_fe['completion_per_watch_time'] = (
            X_fe['completion_rate']
            / (X_fe['avg_watch_time_minutes_per_week'] + 1)
        )
        X_fe['completed_watch_minutes_proxy'] = (
            X_fe['avg_watch_time_minutes_per_week']
            * X_fe['completion_rate']
            / 100
        )
        X_fe['incomplete_watch_minutes_proxy'] = (
            X_fe['avg_watch_time_minutes_per_week']
            * (100 - X_fe['completion_rate'])
            / 100
        )
        X_fe['low_completion_flag'] = (
            X_fe['completion_rate'] <= self.completion_low_q_
        ).astype(int)
        X_fe['high_completion_flag'] = (
            X_fe['completion_rate'] >= self.completion_high_q_
        ).astype(int)

        # 4) Recommendation response: 추천 클릭과 실제 완주 사이의 간극을 본다.
        X_fe['recommendation_completion_gap'] = (
            X_fe['recommendation_click_rate'] - X_fe['completion_rate']
        )
        X_fe['recommendation_completion_ratio'] = (
            X_fe['recommendation_click_rate']
            / (X_fe['completion_rate'] + eps)
        )
        X_fe['low_recommendation_response_flag'] = (
            X_fe['recommendation_click_rate'] <= self.reco_click_low_q_
        ).astype(int)
        X_fe['high_recommendation_response_flag'] = (
            X_fe['recommendation_click_rate'] >= self.reco_click_high_q_
        ).astype(int)
        X_fe['low_interest_flag'] = (
            (X_fe['recommendation_click_rate'] <= self.reco_click_low_q_)
            & (X_fe['completion_rate'] <= self.completion_low_q_)
        ).astype(int)

        # 5) Inactivity risk: 최근 접속 공백이 길고 사용량도 낮으면 캠페인 우선순위가 높다.
        X_fe['inactive_ratio_to_account_age'] = (
            X_fe['days_since_last_login']
            / (X_fe['account_age_months'] * 30 + 1)
        )
        X_fe['high_inactivity_flag'] = (
            X_fe['days_since_last_login'] >= self.inactive_high_q_
        ).astype(int)
        X_fe['low_watch_flag'] = (
            X_fe['avg_watch_time_minutes_per_week'] <= self.watch_time_low_q_
        ).astype(int)
        X_fe['low_session_flag'] = (
            X_fe['watch_sessions_per_week'] <= self.sessions_low_q_
        ).astype(int)
        X_fe['inactive_low_watch_flag'] = (
            (X_fe['high_inactivity_flag'] == 1)
            & (X_fe['low_watch_flag'] == 1)
        ).astype(int)

        # 6) Engagement score: 사용량, 빈도, 완주율, 추천 반응, 비활성 정도를 하나의 점수로 요약한다.
        watch_norm = np.clip(X_fe['avg_watch_time_minutes_per_week'] / self.watch_time_p95_, 0, 1)
        session_norm = np.clip(X_fe['watch_sessions_per_week'] / self.session_p95_, 0, 1)
        completion_norm = np.clip(X_fe['completion_rate'] / 100, 0, 1)
        reco_norm = np.clip(X_fe['recommendation_click_rate'] / 100, 0, 1)
        recency_norm = 1 - np.clip(X_fe['days_since_last_login'] / self.inactive_p95_, 0, 1)

        X_fe['engagement_score'] = 100 * (
            0.25 * watch_norm
            + 0.20 * session_norm
            + 0.25 * completion_norm
            + 0.15 * reco_norm
            + 0.15 * recency_norm
        )
        self.low_engagement_cutoff_ = 40
        X_fe['low_engagement_flag'] = (X_fe['engagement_score'] < self.low_engagement_cutoff_).astype(int)

        # 7) Plan/device interaction: 저가 요금제 + 모바일 중심 사용자는 이탈 액션을 다르게 설계할 수 있다.
        X_fe['basic_mobile_flag'] = (
            (X_fe['subscription_type'].astype(str) == 'Basic')
            & (X_fe['primary_device'].astype(str) == 'Mobile')
        ).astype(int)
        X_fe['basic_inactive_flag'] = (
            (X_fe['subscription_type'].astype(str) == 'Basic')
            & (X_fe['high_inactivity_flag'] == 1)
        ).astype(int)
        X_fe['mobile_low_completion_flag'] = (
            (X_fe['primary_device'].astype(str) == 'Mobile')
            & (X_fe['low_completion_flag'] == 1)
        ).astype(int)

        # 8) Categorical interactions: OneHotEncoder가 처리할 수 있는 조합형 범주 피처.
        X_fe['plan_device_combo'] = (
            X_fe['subscription_type'].astype(str)
            + '__'
            + X_fe['primary_device'].astype(str)
        )
        X_fe['plan_payment_combo'] = (
            X_fe['subscription_type'].astype(str)
            + '__'
            + X_fe['payment_method'].astype(str)
        )
        X_fe['genre_time_combo'] = (
            X_fe['favorite_genre'].astype(str)
            + '__'
            + X_fe['time_of_day'].astype(str)
        )
        X_fe['region_plan_combo'] = (
            X_fe['region'].astype(str)
            + '__'
            + X_fe['subscription_type'].astype(str)
        )
        X_fe['source_genre_combo'] = (
            X_fe['recommendation_source'].astype(str)
            + '__'
            + X_fe['favorite_genre'].astype(str)
        )

        # 9) Risk flags: retention action matrix에서 바로 사용할 수 있는 누적 위험 신호.
        risk_flag_cols = [
            'high_inactivity_flag',
            'low_watch_flag',
            'low_session_flag',
            'low_completion_flag',
            'low_recommendation_response_flag',
            'inactive_low_watch_flag',
            'low_interest_flag',
            'basic_mobile_flag',
            'low_engagement_flag',
        ]
        X_fe['high_risk_behavior_count'] = X_fe[risk_flag_cols].sum(axis=1)
        X_fe['multi_risk_flag'] = (X_fe['high_risk_behavior_count'] >= 3).astype(int)
        X_fe['critical_risk_flag'] = (X_fe['high_risk_behavior_count'] >= 5).astype(int)

        # Campaign action mapping용 해석 피처. 모델에는 OneHotEncoder로 들어가며, 분석 테이블에도 쓸 수 있다.
        X_fe['retention_action_segment'] = np.select(
            [
                X_fe['inactive_low_watch_flag'] == 1,
                X_fe['low_interest_flag'] == 1,
                X_fe['basic_mobile_flag'] == 1,
                X_fe['low_completion_flag'] == 1,
            ],
            [
                'winback_inactive_low_watch',
                'recommendation_reactivation',
                'mobile_basic_offer',
                'content_quality_or_fit',
            ],
            default='monitor_or_standard_campaign',
        )

        return X_fe


**생성 피처 의미**

- `watch_time_per_session`: 한 시청 세션당 평균 시청 시간이다. 짧으면 콘텐츠 몰입도가 낮을 수 있다.
- `watch_time_per_account_age`, `session_per_account_age`: 가입 기간 대비 사용 강도다. 오래된 계정인데 최근 사용량이 낮으면 휴면화 가능성이 있다.
- `completion_per_watch_time`, `completed_watch_minutes_proxy`: 시청량이 실제 완주 행동으로 이어지는지 보는 피처다.
- `recommendation_completion_gap`: 추천 클릭률과 완주율의 차이다. 클릭은 하지만 완주하지 않으면 추천 품질 또는 콘텐츠 만족도 문제가 있을 수 있다.
- `inactive_low_watch_flag`: 최근 미접속과 낮은 시청량이 동시에 나타나는 win-back 우선 타겟 신호다.
- `low_interest_flag`: 추천에도 반응이 낮고 완주도 낮은 고객이다. 추천/콘텐츠 개인화 캠페인 후보로 볼 수 있다.
- `basic_mobile_flag`: Basic 요금제와 Mobile 중심 사용의 조합이다. 가격 민감도 또는 모바일 경험 이슈와 연결해 해석할 수 있다.
- `engagement_score`: 시청량, 세션 빈도, 완주율, 추천 반응, 최근 접속성을 0~100 점수로 요약한 종합 engagement 지표다.
- `low_engagement_flag`: engagement score가 낮은 고객을 표시한다.
- `high_risk_behavior_count`: 여러 위험 flag가 동시에 몇 개나 켜졌는지 세는 피처다. retention 우선순위와 action matrix에 직접 연결하기 쉽다.


## 12. Feature Engineering 적용 전후 성능 비교 코드

아래 코드는 같은 train/test split에서 원본 피처 모델과 feature engineering 모델을 비교한다. 평가는 accuracy가 아니라 risk ranking과 캠페인 타겟팅에 더 가까운 지표를 포함한다.

- PR AUC: churn class ranking 품질을 본다.
- ROC AUC: 전체 ranking 분리도를 본다.
- Precision, Recall, F1: threshold 0.5 기준 분류 성능이다.
- Top 10% churn rate: risk score 상위 10% 안의 실제 churn 비율이다.
- Top 10% lift: 전체 churn rate 대비 상위 10% churn rate가 몇 배 높은지 본다.
- Top 20% / Top 30% captured churner: 전체 churner 중 risk score 상위 k%가 얼마나 많이 포착했는지 본다.


In [6]:
def make_model_candidates():
    """Logistic Regression, RandomForest, GradientBoosting, optional LightGBM 후보를 만든다."""
    candidates = {
        'logistic_regression': LogisticRegression(
            max_iter=2000,
            class_weight='balanced',
            random_state=RANDOM_STATE,
        ),
        'random_forest': RandomForestClassifier(
            n_estimators=400,
            min_samples_leaf=10,
            class_weight='balanced',
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        'gradient_boosting': GradientBoostingClassifier(
            n_estimators=250,
            learning_rate=0.05,
            max_depth=3,
            random_state=RANDOM_STATE,
        ),
    }

    try:
        from lightgbm import LGBMClassifier
        candidates['lightgbm'] = LGBMClassifier(
            n_estimators=500,
            learning_rate=0.03,
            num_leaves=31,
            subsample=0.9,
            colsample_bytree=0.9,
            class_weight='balanced',
            random_state=RANDOM_STATE,
            n_jobs=-1,
            verbose=-1,
        )
    except Exception as exc:
        print('[info] LightGBM unavailable, skipped:', exc)

    return candidates


def build_pipeline_from_train(X_train, estimator, use_feature_engineering=False):
    """Train 데이터 기준으로 ColumnTransformer 컬럼 목록을 안전하게 구성한다."""
    steps = []

    if use_feature_engineering:
        feature_engineer = ChurnBehaviorFeatureEngineer()
        X_for_schema = feature_engineer.fit_transform(X_train)
        steps.append(('feature_engineering', feature_engineer))
    else:
        X_for_schema = X_train.copy()

    preprocessor = build_preprocessor(X_for_schema)
    steps.extend([
        ('preprocess', preprocessor),
        ('model', clone(estimator)),
    ])
    return Pipeline(steps)


def targeting_metrics(y_true, y_score):
    """Risk score 상위 고객군 기준 캠페인 타겟팅 지표를 계산한다."""
    temp = pd.DataFrame({
        'actual_churn': np.asarray(y_true),
        'risk_score': np.asarray(y_score),
    }).sort_values('risk_score', ascending=False).reset_index(drop=True)

    overall_churn_rate = temp['actual_churn'].mean()
    total_churners = temp['actual_churn'].sum()

    top10_n = max(1, int(np.ceil(len(temp) * 0.10)))
    top20_n = max(1, int(np.ceil(len(temp) * 0.20)))
    top30_n = max(1, int(np.ceil(len(temp) * 0.30)))

    top10_churn_rate = temp.head(top10_n)['actual_churn'].mean()
    top20_captured = temp.head(top20_n)['actual_churn'].sum() / total_churners
    top30_captured = temp.head(top30_n)['actual_churn'].sum() / total_churners

    return {
        'top_10pct_churn_rate': top10_churn_rate,
        'top_10pct_lift': top10_churn_rate / overall_churn_rate,
        'top_20pct_captured_churner': top20_captured,
        'top_30pct_captured_churner': top30_captured,
    }


def evaluate_for_campaign(y_true, y_score, threshold=0.5):
    """분류 지표와 risk ranking 지표를 함께 반환한다."""
    y_pred = (np.asarray(y_score) >= threshold).astype(int)
    metrics = {
        'pr_auc': average_precision_score(y_true, y_score),
        'roc_auc': roc_auc_score(y_true, y_score),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'confusion_matrix': confusion_matrix(y_true, y_pred),
    }
    metrics.update(targeting_metrics(y_true, y_score))
    return metrics


def compare_feature_engineering_models(X_train, X_test, y_train, y_test):
    """원본 피처와 feature engineering 피처를 같은 모델 후보군으로 비교한다."""
    rows = []
    confusion_matrices = {}
    fitted_models = {}

    for model_name, estimator in make_model_candidates().items():
        for feature_set, use_fe in [('original', False), ('behavior_fe', True)]:
            print(f'[fit] {model_name} / {feature_set}')
            pipe = build_pipeline_from_train(
                X_train=X_train,
                estimator=estimator,
                use_feature_engineering=use_fe,
            )
            pipe.fit(X_train, y_train)
            y_score = get_scores(pipe, X_test)
            metrics = evaluate_for_campaign(y_test, y_score, threshold=0.5)

            row = {
                'model': model_name,
                'feature_set': feature_set,
                **{k: v for k, v in metrics.items() if k != 'confusion_matrix'},
            }
            rows.append(row)
            confusion_matrices[(model_name, feature_set)] = metrics['confusion_matrix']
            fitted_models[(model_name, feature_set)] = pipe

    comparison = pd.DataFrame(rows).sort_values(
        ['model', 'feature_set']
    ).reset_index(drop=True)
    return comparison, confusion_matrices, fitted_models


fe_comparison, fe_confusion_matrices, fe_fitted_models = compare_feature_engineering_models(
    X_train_original,
    X_test_original,
    y_train,
    y_test,
)

fe_comparison


[fit] logistic_regression / original
[fit] logistic_regression / behavior_fe
[fit] random_forest / original
[fit] random_forest / behavior_fe
[fit] gradient_boosting / original
[fit] gradient_boosting / behavior_fe
[fit] lightgbm / original
[fit] lightgbm / behavior_fe


,model,feature_set,pr_auc,roc_auc,precision,recall,f1,top_10pct_churn_rate,top_10pct_lift,top_20pct_captured_churner,top_30pct_captured_churner
0,gradient_boosting,behavior_fe,0.759272,0.903182,0.759975,0.591495,0.665234,0.858,4.099379,0.668896,0.814620
1,gradient_boosting,original,0.758621,0.903160,0.762829,0.582418,0.660526,0.867,4.142379,0.670330,0.810320
2,lightgbm,behavior_fe,0.756155,0.901412,0.580849,0.791209,0.669903,0.857,4.094601,0.663163,0.810320
3,lightgbm,original,0.757937,0.902859,0.582639,0.801720,0.674844,0.870,4.156713,0.666030,0.812231
4,logistic_regression,behavior_fe,0.761126,0.905543,0.569444,0.822742,0.673051,0.864,4.128046,0.677019,0.820831
5,logistic_regression,original,0.763072,0.906524,0.570385,0.820831,0.673066,0.873,4.171046,0.675585,0.820354
6,random_forest,behavior_fe,0.730519,0.893191,0.611742,0.746775,0.672547,0.827,3.951266,0.657430,0.799331
7,random_forest,original,0.742572,0.897551,0.616679,0.770186,0.684937,0.836,3.994267,0.661730,0.806020


In [7]:
# 모델별 feature engineering 개선폭을 확인한다.
# 캠페인 목적에서는 PR AUC와 top-k ranking 지표를 우선해서 본다.
ranking_metric_cols = [
    'pr_auc',
    'roc_auc',
    'precision',
    'recall',
    'f1',
    'top_10pct_churn_rate',
    'top_10pct_lift',
    'top_20pct_captured_churner',
    'top_30pct_captured_churner',
]

fe_gain_summary = (
    fe_comparison
    .pivot(index='model', columns='feature_set', values=ranking_metric_cols)
)

# behavior_fe - original 형태로 gain table을 만든다.
fe_gain_rows = []
for model_name in fe_comparison['model'].unique():
    original_row = fe_comparison.query("model == @model_name and feature_set == 'original'").iloc[0]
    fe_row = fe_comparison.query("model == @model_name and feature_set == 'behavior_fe'").iloc[0]
    gain = {'model': model_name}
    for col in ranking_metric_cols:
        gain[f'{col}_gain'] = fe_row[col] - original_row[col]
    fe_gain_rows.append(gain)

fe_gain_df = pd.DataFrame(fe_gain_rows)
fe_gain_df


,model,pr_auc_gain,roc_auc_gain,precision_gain,recall_gain,f1_gain,top_10pct_churn_rate_gain,top_10pct_lift_gain,top_20pct_captured_churner_gain,top_30pct_captured_churner_gain
0,gradient_boosting,0.000651,0.000022,-0.002853,0.009078,0.004708,-0.009,-0.043000,-0.001433,0.004300
1,lightgbm,-0.001782,-0.001448,-0.001790,-0.010511,-0.004941,-0.013,-0.062112,-0.002867,-0.001911
2,logistic_regression,-0.001946,-0.000982,-0.000941,0.001911,-0.000015,-0.009,-0.043000,0.001433,0.000478
3,random_forest,-0.012053,-0.004360,-0.004938,-0.023411,-0.012390,-0.009,-0.043000,-0.004300,-0.006689


## 14. 실험 결과 해석

이번 feature engineering은 **모델 성능을 크게 개선하기보다는, churn risk를 설명하고 retention action으로 연결하기 쉬운 행동 신호를 만드는 데 더 강점이 있다.**

전체 비교 결과를 보면 `behavior_fe`가 모든 모델에서 일관되게 좋아지지는 않았다. 특히 이 프로젝트의 핵심 지표인 PR AUC, top 10% churn rate, top 10% lift 기준에서는 원본 피처가 대체로 더 안정적이다.

모델별로 보면 다음과 같다.

- **GradientBoosting**은 `behavior_fe` 적용 후 PR AUC가 0.7586에서 0.7593으로 아주 소폭 상승했고, recall과 F1도 개선됐다. 다만 top 10% churn rate는 86.7%에서 85.8%로 낮아졌고, top 10% lift도 하락했다. 즉 threshold 분류 성능은 약간 좋아졌지만, 캠페인 상위 10% 타겟팅 관점에서는 개선이라고 보기 어렵다.
- **Logistic Regression**은 recall이 약간 상승했지만 PR AUC, ROC AUC, F1은 거의 동일하거나 소폭 하락했다. top 20%와 top 30% captured churner는 아주 작게 개선됐지만, top 10% churn rate와 lift는 하락했다.
- **LightGBM**은 `behavior_fe` 적용 후 PR AUC, ROC AUC, recall, F1, top-k 지표가 대부분 하락했다.
- **RandomForest**는 가장 뚜렷하게 악화됐다. PR AUC가 0.7426에서 0.7305로 낮아졌고, recall/F1/top-k captured churner도 모두 하락했다.

따라서 현재 결과만 기준으로는 **behavior feature engineering을 최종 예측 모델의 기본 feature set으로 채택하기는 어렵다.**


## 15. Campaign Targeting 관점 해석

이 프로젝트의 목표는 단순 classification accuracy가 아니라 **risk score 상위 고객을 얼마나 잘 우선순위화하는가**이다. 이 기준에서는 top 10% churn rate와 lift가 특히 중요하다.

실험 결과에서 top 10% churn rate는 다음과 같이 원본 피처가 더 높다.

- Logistic Regression: original 87.3% vs behavior_fe 86.4%
- LightGBM: original 87.0% vs behavior_fe 85.7%
- GradientBoosting: original 86.7% vs behavior_fe 85.8%
- RandomForest: original 83.6% vs behavior_fe 82.7%

즉 캠페인 예산이 제한되어 상위 10% 고객만 타겟팅해야 한다면, 현재 feature engineering은 상위 위험군의 churn 밀도를 높이지 못했다. retention campaign의 1차 타겟 선정 모델로는 원본 feature set 또는 이전 Stacking 모델을 유지하는 편이 더 합리적이다.

다만 `behavior_fe`가 완전히 의미 없다는 뜻은 아니다. 새로 만든 피처들은 예측 성능보다 **타겟 고객을 설명하고 action matrix로 연결하는 데 유용하다.** 예를 들어 risk score 상위 고객 중에서도 다음과 같이 캠페인 메시지를 나눌 수 있다.

- `inactive_low_watch_flag = 1`: 장기 미접속 + 낮은 시청량 고객이므로 win-back 쿠폰, 복귀 알림, 최근 인기 콘텐츠 추천이 적합하다.
- `low_interest_flag = 1`: 추천 클릭률과 완주율이 모두 낮으므로 추천 품질 개선, 장르 재탐색, 개인화 onboarding 메시지가 적합하다.
- `basic_mobile_flag = 1`: Basic + Mobile 고객이므로 모바일 최적화 혜택, 저가 요금제 유지 혜택, 짧은 콘텐츠 추천이 적합하다.
- `high_risk_behavior_count`가 높음: 여러 위험 행동이 동시에 나타나는 고객이므로 일반 캠페인보다 강한 retention incentive를 우선 검토할 수 있다.

따라서 현재 결론은 **모델 입력 피처로는 보수적으로 사용하고, 캠페인 세그먼트 해석 피처로는 적극 활용하는 것**이 좋다.


## 16. 최종 판단

현재 `07-2` 실험 기준 최종 판단은 다음과 같다.

**최종 예측 모델 관점**

- `behavior_fe`는 전체 모델에서 일관된 성능 개선을 만들지 못했다.
- PR AUC와 top 10% lift가 대체로 하락했기 때문에, 최종 risk score 산출 모델에는 원본 feature set 또는 이전 Stacking 결과를 우선 사용하는 것이 적절하다.
- GradientBoosting에서는 F1과 recall이 소폭 개선됐지만, 캠페인 핵심 지표인 top 10% churn rate가 하락했으므로 최종 모델 교체 근거로는 약하다.

**Retention action 관점**

- `inactive_low_watch_flag`, `low_interest_flag`, `basic_mobile_flag`, `low_engagement_flag`, `high_risk_behavior_count`는 action matrix에 직접 연결 가능하다.
- 따라서 최종 모델이 원본 피처 기반이더라도, risk score 산출 후 상위 decile 고객을 설명하고 campaign type을 배정하는 보조 피처로 유지할 가치가 있다.

**추천 운영 방식**

1. 최종 churn risk score는 성능이 더 안정적인 원본/Stacking 계열 모델로 산출한다.
2. `ChurnBehaviorFeatureEngineer`는 상위 risk 고객의 사후 profiling과 retention action matrix 생성에 사용한다.
3. 모델 성능 개선을 다시 시도한다면 전체 피처를 무조건 추가하기보다, `high_risk_behavior_count`, `engagement_score`, `inactive_low_watch_flag`처럼 해석 가능한 소수 피처만 넣는 ablation test를 진행한다.
4. 최종 판단 기준은 accuracy가 아니라 PR AUC, top 10% lift, top 20/30% captured churner의 개선 여부로 둔다.

요약하면, 이번 feature engineering은 **예측 모델 개선에는 실패에 가깝지만, retention campaign 설계용 해석 피처로는 성공적인 실험**이다.


In [8]:
# Confusion matrix는 threshold 0.5 기준이다.
# 예: Logistic Regression 원본 vs feature engineering 비교
for key, cm in fe_confusion_matrices.items():
    print(key)
    print(cm)
    print()


('logistic_regression', 'original')
[[6613 1294]
 [ 375 1718]]

('logistic_regression', 'behavior_fe')
[[6605 1302]
 [ 371 1722]]

('random_forest', 'original')
[[6905 1002]
 [ 481 1612]]

('random_forest', 'behavior_fe')
[[6915  992]
 [ 530 1563]]

('gradient_boosting', 'original')
[[7528  379]
 [ 874 1219]]

('gradient_boosting', 'behavior_fe')
[[7516  391]
 [ 855 1238]]

('lightgbm', 'original')
[[6705 1202]
 [ 415 1678]]

('lightgbm', 'behavior_fe')
[[6712 1195]
 [ 437 1656]]



In [9]:
# Feature engineering 후 실제로 어떤 컬럼이 추가되는지 확인한다.
fe_debugger = ChurnBehaviorFeatureEngineer()
X_train_behavior_fe = fe_debugger.fit_transform(X_train_original)
behavior_added_cols = sorted(set(X_train_behavior_fe.columns) - set(X_train_original.columns))

print('Original feature count:', X_train_original.shape[1])
print('Behavior FE feature count:', X_train_behavior_fe.shape[1])
print('Added feature count:', len(behavior_added_cols))
behavior_added_cols


Original feature count: 18
Behavior FE feature count: 53
Added feature count: 35


['basic_inactive_flag',
 'basic_mobile_flag',
 'completed_watch_minutes_proxy',
 'completion_per_watch_time',
 'critical_risk_flag',
 'engagement_score',
 'genre_time_combo',
 'high_completion_flag',
 'high_inactivity_flag',
 'high_recommendation_response_flag',
 'high_risk_behavior_count',
 'inactive_low_watch_flag',
 'inactive_ratio_to_account_age',
 'incomplete_watch_minutes_proxy',
 'low_completion_flag',
 'low_engagement_flag',
 'low_interest_flag',
 'low_recommendation_response_flag',
 'low_session_flag',
 'low_watch_flag',
 'mobile_low_completion_flag',
 'multi_risk_flag',
 'plan_device_combo',
 'plan_payment_combo',
 'recommendation_completion_gap',
 'recommendation_completion_ratio',
 'region_plan_combo',
 'retention_action_segment',
 'session_count_per_week_session',
 'session_per_account_age',
 'source_genre_combo',
 'watch_time_per_account_age',
 'watch_time_per_session',
 'weekly_session_per_account_age',
 'weekly_watch_session_intensity']

## 13. Feature Engineering 효과 판단 기준

이 feature engineering이 실제로 도움이 됐다고 판단하려면 accuracy보다 churn risk ranking 지표가 개선되어야 한다.

**도움이 됐다고 볼 수 있는 경우**

- PR AUC가 원본 대비 상승한다. churn class가 불균형이면 ROC AUC보다 PR AUC 개선을 더 중요하게 본다.
- Top 10% churn rate와 Top 10% lift가 상승한다. 캠페인 예산이 제한될수록 이 지표가 중요하다.
- Top 20% 또는 Top 30% captured churner가 상승한다. 상위 타겟군이 실제 churner를 더 많이 포함한다는 뜻이다.
- Recall이 유지되거나 상승하면서 precision/F1이 크게 훼손되지 않는다.
- `high_risk_behavior_count`, `inactive_low_watch_flag`, `low_interest_flag`, `basic_mobile_flag` 같은 피처가 상위 risk 고객 설명과 retention action matrix에 자연스럽게 연결된다.

**도움이 부족하다고 봐야 하는 경우**

- PR AUC, Top 10% lift, captured churner가 모두 비슷하거나 하락한다.
- 분류 threshold 0.5의 F1만 약간 올랐지만 상위 risk ranking 지표가 악화된다.
- 모델별로 개선 방향이 일관되지 않고 특정 모델에서만 우연히 좋아진다.
- 새 피처가 비즈니스 액션으로 해석되지 않는다. 예를 들어 점수는 올랐지만 어떤 캠페인을 해야 하는지 설명할 수 없으면 이 프로젝트 목표와 맞지 않는다.

**이 프로젝트에서의 우선순위**

1. `top_10pct_churn_rate`와 `top_10pct_lift` 개선
2. `top_20pct_captured_churner`, `top_30pct_captured_churner` 개선
3. PR AUC 개선
4. Recall과 F1의 균형 유지
5. action matrix로 설명 가능한 위험 피처 확보

따라서 최종 모델 선택은 단순 accuracy가 아니라, “상위 risk score 고객을 뽑았을 때 실제 churner가 얼마나 밀집되는가”를 기준으로 결정하는 것이 적절하다.
